<a href="https://colab.research.google.com/github/bookwork123/MachineLearning/blob/main/HW5/analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**1. Explain how implicit missing values impact medical datasets, and justify your choice of imputation strategy when handling these zeros compared to outright row deletion.**

Implicit missing values are dangerous in medical datasets because they seem like real measurements. In the Pima dataset, missing values were recorded as 0, so info() reported 768 non-null values in every column and isnull() found nothing. Only describe() was useful: Glucose, BloodPressure, SkinThickness, Insulin, and BMI all had a minimum of 0, which is impossible for an actual patient. If these zeros are left in, the model treats them as real values. For example, a patient with a "glucose of 0" looks extremely healthy, which pulls the model's idea of a normal patient in the wrong direction. The zeros also drag down column averages and widen the spread that StandardScaler uses, so every other patient's scaled values end up becoming more off. The problem was large here: in Task 3, I found 374 zeros in Insulin (48.7% of patients) and 227 in SkinThickness (29.6%), along with smaller numbers in BloodPressure (35), BMI (11), and Glucose (5).

I filled the gaps instead of dropping rows, because dropping would throw away a huge part of an already small dataset. 376 of the 768 patients have at least one missing value, so deleting them would leave only 392 patients, which is almost half the data gone. It would also throw away good readings, since most of those patients are only missing Insulin or SkinThickness, while their Glucose, BMI, and Age are fine. Those are the most important features for predicting diabetes, so losing them would hurt the model more than imperfect fill values would.

I used the median to fill the gaps, because Insulin and SkinThickness are skewed. A few patients have very high readings (Insulin goes up to 846), which pull the mean away from a typical value but barely change the median. Before imputing, I replaced the zeros with NaN so SimpleImputer would recognize them as missing. Because I filled the gaps in Task 3, before splitting the data, the medians were calculated from all 768 patients, including the ones later used for testing. This is a small amount of data leakage, but a median barely changes when a few patients are added or removed, so it had very little effect on my results. The main drawback is that almost half of the Insulin column is now the same filled-in value, so what the model learns from Insulin is less reliable than what it learns from the more complete columns.

**2. Contrast how hard voting aggregates predictions versus soft voting. Why do well-calibrated probability estimates matter when implementing soft voting across heterogeneous models?**

Hard voting only looks at each model's final answer. Logistic Regression, the SVC, and the Decision Tree each predict "tested_positive" or "tested_negative", and whichever label gets at least two of the three votes wins. It doesn't matter if a model was 51% sure or 99% sure, since every vote counts the same. Soft voting looks at how confident each model is. Each model gives a probability that the patient has diabetes, the three probabilities are averaged, and the class with the higher average wins. This means one very confident model can outweigh two models that are barely leaning the other way.

Well-calibrated probabilities matter because soft voting assumes that a probability means the same thing from every model. A model is well-calibrated if, out of all the patients it gives a 70% chance, about 70% actually have diabetes. My three models are very different (heterogeneous), and so are their probabilities. Logistic Regression produces probabilities naturally from its formula, and the SVC's probabilities are estimated in an extra step after training. The Decision Tree's probabilities are just the share of diabetic patients in the leaf a patient lands in. A poorly calibrated, overconfident model can end up dominating the vote, which defeats the purpose of combining different models.

**3. What is the role of out-of-bag evaluation in Bagging classifiers, and why does an OOB score serve as an efficient alternative to traditional cross-validation splits during ensemble training?**

In bagging, each of the 500 decision trees is trained on a random sample of the training set drawn with replacement. Because some patients get picked more than once, each tree only sees about 63% of the unique training patients. The other 37% are that tree's out-of-bag (OOB) patients. Since the tree never saw them, they work like a small test set for that tree. OOB evaluation predicts each training patient using only the trees that didn't train on it, then checks how many of those predictions were correct. The result is the OOB score, an estimate of how well the ensemble will do on new patients without using the test set at all.

The OOB score is an efficient alternative to cross-validation because it comes almost for free. With 5-fold cross-validation, I would have to train the whole 500-tree ensemble five separate times, once for each fold, which is 2,500 trees. The OOB score uses the left-out patients that bagging already creates, so it only needs the one training run. It also doesn't require setting aside a separate validation set, which matters with a small dataset like this one, where every patient is valuable for training.

**4. How does introducing random feature subsetting in Random Forests differ from standard Bagging, and how does this feature diversity impact model variance and generalization?**

Standard bagging only adds randomness through the data. Each tree gets a different bootstrap sample, but at every split, it can still look at all 8 features and pick the best one. A Random Forest adds a second layer of randomness: at each split, the tree can only choose from a random subset of the features. I used max_features="sqrt", so each split only looks at about 2 of the 8 features.

Diversity reduces variance because averaging only cancels out errors when the trees make different errors. If all 500 trees make the same mistakes, averaging them doesn't help much. If their mistakes are different, they cancel out, so the forest's predictions are more stable and less dependent on the particular patients in the training set, which should help it generalize to new patients.

**5. Discuss the mechanics of sequential sample weighting in AdaBoost. Based on your training versus test accuracy results, did AdaBoost exhibit signs of overfitting, and how can hyperparameter adjustments mitigate this?**

AdaBoost trains its models one at a time, and each new model focuses on the patients the previous models got wrong. At the start, every training patient has the same weight. The first decision stump (a tree with only one split) is trained, and AdaBoost measures its weighted error. The stump gets its own weight based on that error, so more accurate stumps get a bigger say in the final vote. Then the weights of the misclassified patients are increased and the weights of the correctly classified patients are decreased, so the next stump pays more attention to the hard-to-classify patients. This repeats for all 200 stumps, and the final prediction is a weighted vote of every stump.

In Task 8, AdaBoost reached 0.8078 accuracy on the training set and 0.7338 on the test set. That is a gap of about 7 points, so the model shows mild signs of overfitting, where it fits the training patients better than it generalizes to new ones.

Several hyperparameter changes could reduce this. Lowering the learning_rate further (for example, to 0.1) makes each stump contribute less, which keeps the model from adjusting too quickly to individual patients. Reducing n_estimators stops the boosting before it starts fitting noise in the training data. Keeping the base estimators as stumps (max_depth=1) instead of deeper trees keeps each learner weak and simple.